# A robustez v2, contada do começo ao fim

Este notebook acompanha a robustez refeita com o protocolo de degradação v2 (`codigo/robustez-2026-11/`).
Ele serve para eu explorar cada resultado e, ao mesmo tempo, contar a história em ordem, com os gráficos que vão para a conversa com a orientação e para o artigo.

Duas regras valem do início ao fim:

- **nenhum número escrito à mão**: toda frase que cita um valor é gerada por uma célula de código, a partir das tabelas do pipeline;
- **nada de análise refeita aqui**: as contas estatísticas vêm de `analise/estatistica-2026-08/tabelas_v2/` (gerado por `run_all.py --v2`). O notebook lê, junta e desenha; as únicas contas novas são descritivas (medianas de caminho) e o IC do ganho estático, feito com a mesma função `bootstrap_bca_geom` do pipeline.

Rode de cima a baixo com o kernel do `.venv` de `analise/estatistica-2026-08/`.

In [ ]:
import os, subprocess, sys, time
from pathlib import Path
import numpy as np, pandas as pd, duckdb
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, TwoSlopeNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from IPython.display import Markdown, display, Image

INICIO = time.time()

def acha_raiz():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "maps").is_dir() and (q / "codigo" / "robustez-2026-11").is_dir():
            return q
    raise SystemExit("abra o notebook de dentro do repositorio")

RAIZ = acha_raiz()
ANALISE = RAIZ / "analise" / "estatistica-2026-08"
T1, T2, T2F = ANALISE / "tabelas", ANALISE / "tabelas_v2", ANALISE / "tabelas_v2_fora_treino"
MEDICOES = ANALISE / "medicoes"
V2 = Path(os.environ.get("HEURISTICAS_DADOS_V2", RAIZ / "dados" / "brutos" / "v2"))
CACHE = V2 / "cache"
FIG = RAIZ / "analise" / "notebook-2026-10" / "figuras"
FIG.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

# dados brutos de marco: HEURISTICAS_DADOS, depois dados/brutos, depois ~/Downloads
def acha_marco():
    for c in (os.environ.get("HEURISTICAS_DADOS"), RAIZ / "dados" / "brutos", Path.home() / "Downloads"):
        if c and (Path(c) / "resultados_robustez.csv").exists():
            return Path(c)
    raise SystemExit("defina HEURISTICAS_DADOS com a pasta dos CSVs de marco (ver dados/MANIFESTO.md)")

MARCO = acha_marco()
sys.path.insert(0, str(ANALISE / "scripts"))
from comum import media_geometrica, bootstrap_bca_geom, categoria, SEED
print("repositorio:", RAIZ)
print("dados v2:   ", V2)
print("dados marco:", MARCO)

In [ ]:
ESTRATEGIAS = ["Manhattan", "Fórmula", "Memória-10", "Memória-20", "Memória-50", "Memória-100"]
MEMORIAS = ESTRATEGIAS[2:]
PADRAO_PT = {"Linear": "Linear", "Organic": "Orgânico", "Radial": "Radial", "Sparse": "Esparso",
             "Stochastic": "Estocástico"}
REGIME_PT = {"com_atualizacao": "com atualização", "sem_atualizacao": "sem atualização"}
GRUPOS = ["Arena", "Área aberta", "Dungeon", "Natureza"]

# paleta Okabe-Ito para destaque, uma familia de azuis para a memoria, cinza para a Manhattan
COR = {"Manhattan": "#7a7a7a", "Fórmula": "#D55E00", "Memória-10": "#8fbfe0", "Memória-20": "#4f9bd0",
       "Memória-50": "#1f6fb2", "Memória-100": "#08306b"}
MARCA = {"Manhattan": "s", "Fórmula": "D", **{m: "o" for m in MEMORIAS}}

def estilo(e, regime="com_atualizacao"):
    sem = e in MEMORIAS and regime == "sem_atualizacao"
    return dict(color=COR[e], marker=MARCA[e], ls="--" if sem else "-", alpha=0.75 if sem else 1.0,
                markerfacecolor="white" if sem else COR[e], markeredgecolor=COR[e])

def rotulo(e, regime="com_atualizacao"):
    if e in MEMORIAS:
        return f"{e} {REGIME_PT[regime]}"
    return e

def num(x, casas=2):
    return f"{x:.{casas}f}".replace(".", ",")

def milhar(n):
    return f"{int(n):,}".replace(",", ".")

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 10, "axes.titlesize": 11.5,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": False, "legend.frameon": False,
    "pdf.fonttype": 42, "svg.hashsalt": "historia_v2",
})

def salvar(fig, nome):
    fig.savefig(FIG / f"{nome}.png", dpi=300, bbox_inches="tight", metadata={"Software": None})
    fig.savefig(FIG / f"{nome}.pdf", bbox_inches="tight", metadata={"CreationDate": None})
    print(f"exportada: figuras/{nome}.png e .pdf")

def espalha(ys, gap):
    """Posicoes de rotulo com distancia minima gap, preservando a ordem."""
    ys = np.asarray(ys, float)
    ordem = np.argsort(ys)
    pos = ys[ordem].copy()
    for i in range(1, len(pos)):
        pos[i] = max(pos[i], pos[i - 1] + gap)
    out = np.empty_like(pos)
    out[ordem] = pos
    return out

In [ ]:
le = lambda pasta, nome: pd.read_csv(pasta / f"b1_{nome}.csv", dtype={"nivel": str})
G, GF = le(T2, "global"), le(T2F, "global")
PM, PMF = le(T2, "por_mapa"), le(T2F, "por_mapa")
POSTOS, FRIEDMAN, CLIFF = le(T2, "postos"), le(T2, "friedman"), le(T2, "cliff")
PN = le(T2, "padrao_nivel")
INTER = pd.read_csv(T2 / "b1_interacao.csv")
SOB = pd.read_csv(T2 / "b1_sobrevivencia_v1_v2.csv")
REJ = pd.read_csv(T2 / "b1_rejeicao.csv")
PIV = pd.read_csv(T2 / "b1_pivos_bloqueados.csv")
COLAPSO = pd.read_csv(T2 / "b1_colapso_pivo.csv")
ANOVA_V2 = {(m, r): pd.read_csv(T2 / f"b1_anova_{m}_{r}.csv", index_col=0)
            for m in ("delta", "absoluta") for r in REGIME_PT}
ANOVA_V1 = pd.read_csv(T1 / "a1_anova_completa.csv", index_col=0)
POSTOS_V1 = pd.read_csv(T1 / "a2_postos.csv", index_col=0)
DELTA_V1 = pd.read_csv(T1 / "a2_escore_degradado.csv", index_col=0)
RESUMO = pd.read_csv(MEDICOES / "degradacao_v2_resumo.csv")

# cache Parquet dos dados brutos v2 (sem as colunas de tempo), gerado uma vez fora do Git
PQ = CACHE / "robustez_v2.parquet"
if not PQ.exists():
    t = time.time()
    duckdb.sql(f"""COPY (SELECT COLUMNS(lambda c: NOT starts_with(c, 'tempo_'))
                   FROM read_csv('{V2 / "resultados_robustez_v2.csv"}', header = true))
                   TO '{PQ}' (FORMAT parquet, COMPRESSION zstd)""")
    print(f"cache criado em {time.time() - t:.0f} s: {PQ}")
v2 = lambda sql: duckdb.sql(sql.replace("v2", f"read_parquet('{PQ}')")).df()

# grids do den501d s42 a 30% para os exemplos visuais
GRIDS = CACHE / "grids"
if not (GRIDS / "den501d_42_Radial_v2_30.txt").exists():
    subprocess.run(["make", "-C", str(RAIZ / "codigo" / "robustez-2026-11"), "grids", f"SAIDA={V2}"],
                   check=True, capture_output=True)

def grid(nome):
    return np.array([list(l) for l in (GRIDS / f"{nome}.txt").read_text().split("\n") if l])

def mapa(nome):
    L = (RAIZ / "maps" / f"{nome}.map").read_text().split("\n")
    h, w = int(L[1].split()[1]), int(L[2].split()[1])
    return np.array([[c == "." for c in L[4 + i][:w]] for i in range(h)])

N_MAPAS = PM[(PM.nivel == "estatico")].mapa.nunique()
N_MAPAS_F = PMF[(PMF.nivel == "estatico")].mapa.nunique()
print("mapas:", N_MAPAS, "| fora do treino:", N_MAPAS_F)

## Ato 0 — A pergunta

Quero saber **qual heurística aguenta melhor um mapa que muda**. Comparo três heurísticas do A\* em mapas de Dragon Age: Origins que vão sendo bloqueados:

- **Manhattan**: não sabe nada do mapa, só a distância em linha reta (em grade);
- **Fórmula**: uma expressão sintetizada por algoritmo genético para cada mapa, que também não guarda nada do mapa;
- **Memória**: guarda as distâncias de alguns pontos fixos (pivôs, com 10, 20, 50 ou 100 deles) até todas as células.

A memória sob degradação tem dois regimes, e eles são o centro da história:

- **com atualização**: as distâncias dos pivôs são recalculadas no mapa degradado;
- **sem atualização**: a memória continua usando as distâncias do mapa original.

In [ ]:
gl = G[(G.metrica == "absoluta") & (G.nivel == "todos")].set_index(["regime", "estrategia"])
ganho = lambda r, e: 1 / gl.loc[(r, e), "media_geometrica"]
pmd = PM[(PM.metrica == "absoluta") & (PM.nivel == "todos")]
sem = pmd[pmd.regime == "sem_atualizacao"].pivot(index="mapa", columns="estrategia", values="valor")
com = pmd[pmd.regime == "com_atualizacao"].pivot(index="mapa", columns="estrategia", values="valor")
venc_sem = int((sem.idxmin(axis=1) == "Fórmula").sum())
venc_com = int((com.idxmin(axis=1) == "Memória-100").sum())
display(Markdown(
    f"**O resultado em uma frase:** num mapa que muda, quem vence depende de a heurística poder ser atualizada. "
    f"Com atualização, a Memória-100 faz {num(ganho('com_atualizacao', 'Memória-100'))}× menos expansões que a "
    f"Manhattan e é a melhor em {venc_com} de {N_MAPAS} mapas. Sem atualização, ela cai para "
    f"{num(ganho('sem_atualizacao', 'Memória-100'))}×, e a fórmula, com {num(ganho('sem_atualizacao', 'Fórmula'))}×, "
    f"passa a ser a melhor em {venc_sem} de {N_MAPAS} mapas."))

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 2.2))
itens = [("Manhattan", "com_atualizacao", "não sabe nada do mapa"),
         ("Fórmula", "com_atualizacao", "sintetizada por AG, sem memória"),
         ("Memória-10", "com_atualizacao", "memória, distâncias recalculadas"),
         ("Memória-100", "com_atualizacao", ""),
         ("Memória-10", "sem_atualizacao", "memória, distâncias do original"),
         ("Memória-100", "sem_atualizacao", "")]
for i, (e, r, txt) in enumerate(itens):
    col, lin = i // 2, i % 2
    x0, y = col * 2.45, 1 - lin * 0.55
    ax.plot([x0, x0 + 0.5], [y, y], lw=2.2, ms=6, **estilo(e, r))
    ax.text(x0 + 0.62, y, rotulo(e, r), va="center", fontsize=9)
    if txt:
        ax.text(x0, 1.45, txt, fontsize=8.5, color="0.35")
ax.text(4.9 + 0.62, 0.15, "10, 20, 50 e 100 pivôs: do claro ao escuro", fontsize=8.5, color="0.35")
ax.set_xlim(-0.1, 7.6); ax.set_ylim(0, 1.7); ax.axis("off")
ax.set_title("Como ler os gráficos: cada heurística tem cor e marcador fixos; tracejado = memória sem atualização")
plt.show()

## Ato 1 — Os dados

**Pergunta:** em que mapas e com que problemas eu testo?

Um **par** é um problema de busca: uma origem e um destino tirados do cenário oficial do Moving AI para aquele mapa. Para cada par, o A\* roda uma vez com cada heurística, e eu conto as **expansões**: quantos nós a busca tirou da fila antes de achar o caminho. Menos expansões = heurística que guia melhor. É a métrica principal porque é determinística (o tempo de execução não é confiável aqui, ver P3).

In [ ]:
pares = RESUMO[(RESUMO.padrao == "Linear") & (RESUMO.nivel == 0.1)].groupby("mapa").pares_validos.first()
mapas = sorted(pares.index, key=lambda m: (GRUPOS.index(categoria(m)), m))
por_grupo = {g: [m for m in mapas if categoria(m) == g] for g in GRUPOS}
ncol = max(len(v) for v in por_grupo.values())
fig, axes = plt.subplots(len(GRUPOS), ncol, figsize=(1.55 * ncol, 1.9 * len(GRUPOS)))
for i, g in enumerate(GRUPOS):
    for j in range(ncol):
        ax = axes[i, j]; ax.axis("off")
        if j < len(por_grupo[g]):
            m = por_grupo[g][j]
            ax.imshow(mapa(m), cmap=ListedColormap(["#2b2b2b", "#f4f1ea"]), interpolation="nearest")
            ax.set_title(f"{m}\n{milhar(pares[m])} pares", fontsize=8, fontweight="normal",
                         loc="center")
    axes[i, 0].text(-0.15, 0.5, g, transform=axes[i, 0].transAxes, rotation=90, ha="right", va="center",
                    fontsize=10, fontweight="bold")
fig.suptitle(f"Os {len(mapas)} mapas, por perfil (escuro = obstáculo)", x=0.02, ha="left",
             fontweight="bold", fontsize=11.5)
fig.tight_layout()
salvar(fig, "mapas_por_perfil")
plt.show()

In [ ]:
sementes = RESUMO.semente.nunique(); padroes = RESUMO.padrao.nunique(); niveis = RESUMO.nivel.nunique()
n_pares = int(pares.sum())
linhas = int(v2("SELECT count(*) AS n FROM v2").n[0])
display(Markdown(
    f"São {len(mapas)} mapas, com {milhar(n_pares)} pares conectados por semente ({milhar(n_pares * sementes)} no "
    f"total, somando as {sementes} sementes). Cada par é degradado por {padroes} padrões em {niveis} níveis "
    f"({', '.join(f'{int(100 * n)}%' for n in sorted(RESUMO.nivel.unique()))}), e a memória roda com "
    f"{v2('SELECT count(DISTINCT n_pivos) AS n FROM v2').n[0]} números de pivôs. "
    f"O arquivo bruto tem {milhar(linhas)} linhas: uma por par × padrão × nível × número de pivôs."))

## Ato 2 — O mapa estático

**Pergunta:** antes de bloquear qualquer coisa, quem guia melhor a busca?

Meço o **ganho sobre a Manhattan**: expansões da Manhattan ÷ expansões da heurística, no mesmo par. Um ganho de 3× quer dizer três vezes menos trabalho. Agrego por média geométrica, primeiro dentro de cada mapa, depois entre os mapas (o mapa é a unidade).

In [ ]:
est = PM[(PM.metrica == "absoluta") & (PM.nivel == "estatico")].pivot(index="mapa", columns="estrategia",
                                                                       values="valor")
ganho_est = 1 / est[ESTRATEGIAS]
ic_est = {e: tuple(bootstrap_bca_geom(ganho_est[e].values)) if e != "Manhattan" else (1.0, 1.0)
          for e in ESTRATEGIAS}
gm_est = {e: media_geometrica(ganho_est[e]) for e in ESTRATEGIAS}
ordem = sorted(ESTRATEGIAS[1:], key=lambda e: gm_est[e])
lider = ordem[-1]
fig, ax = plt.subplots(figsize=(7.2, 3.6))
for i, e in enumerate(ordem):
    y = ganho_est[e].sort_values().values
    x = i + np.linspace(-0.18, 0.18, len(y))
    ax.scatter(x, y, s=14, color=COR[e], alpha=0.45, marker=MARCA[e], lw=0)
    lo, hi = ic_est[e]
    ax.plot([i, i], [lo, hi], color="black", lw=1.4)
    ax.plot(i, gm_est[e], marker=MARCA[e], ms=9, color=COR[e], mec="black", mew=0.8)
    ax.text(i + 0.26, gm_est[e], f"{num(gm_est[e])}×", va="center", fontsize=9)
ax.axhline(1, color=COR["Manhattan"], lw=1, ls=":")
ax.text(-0.45, 1.04, "Manhattan = 1×", color=COR["Manhattan"], ha="left", fontsize=8.5)
ax.set_yscale("log"); ax.set_xticks(range(len(ordem)), ordem)
ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g}×".replace(".", ",")))
ax.set_ylabel("ganho sobre a Manhattan (escala log)")
ax.set_title(f"No mapa estático, a {lider} vence: {num(gm_est[lider])}× menos expansões que a Manhattan")
ax.text(0, -0.2, f"marcador grande: média geométrica nos {N_MAPAS} mapas; barra: IC 95% BCa; pontos: um por mapa",
        transform=ax.transAxes, fontsize=8, color="0.35")
salvar(fig, "ganho_estatico")
plt.show()

In [ ]:
display(Markdown(
    f"**O que isso significa.** Sem degradação, a memória vence: a ordem é "
    f"{' > '.join(reversed(ordem))} > Manhattan. A fórmula ganha da Manhattan "
    f"({num(gm_est['Fórmula'])}×, IC {num(ic_est['Fórmula'][0])}–{num(ic_est['Fórmula'][1])}), "
    f"mas fica atrás de qualquer memória. É o ponto de partida que a degradação vai pôr à prova."))

## Ato 3 — O problema do protocolo antigo

**Pergunta:** por que o resultado do WPerformance (v1) precisava ser refeito?

No v1, o gerador de bloqueios podia cair em cima da origem ou do destino de um par, ou partir o mapa em pedaços. Os pares que ficavam sem caminho eram descartados, e os que sobravam eram escolhidos uma vez só, a 30%, e reaproveitados nos outros níveis (P2). No v2, origem e destino são protegidos (R1) e nenhum bloqueio pode dividir o mapa (R2).

In [ ]:
sob = SOB.assign(pct_v1=100 * SOB.taxa_v1, pct_v2=100 * SOB.taxa_v2)
assert (sob.pct_v2 == 100).all() and (sob.pct_v1 < 100).all()
pior = sob[sob.nivel == 0.3].sort_values("pct_v1").iloc[0]
fig, ax = plt.subplots(figsize=(7.2, 4.2))
niv = sorted(sob.nivel.unique())
xs = [int(100 * n) for n in niv]
cinzas = dict(zip(PADRAO_PT, ["0.1", "0.27", "0.38", "0.48", "0.58"]))
marcas = dict(zip(PADRAO_PT, ["o", "s", "^", "v", "D"]))
fim = []
for p in PADRAO_PT:
    y = sob[sob.padrao == p].sort_values("nivel").pct_v1.values
    ax.plot(xs, y, color=cinzas[p], marker=marcas[p], lw=1.6)
    fim.append((p, y[-1]))
pos = 10 ** espalha(np.log10([v for _, v in fim]), 0.12)
for (p, v), yy in zip(fim, pos):
    ax.text(xs[-1] + 1, yy, f"{PADRAO_PT[p]}: {num(v)}%", va="center", fontsize=8.5, color=cinzas[p])
ax.plot(xs, [100] * len(xs), color="#009E73", lw=3, marker="o")
ax.text(xs[-1] + 1, 100, "v2, todos os padrões: 100%", va="center", fontsize=9, color="#009E73",
        fontweight="bold")
ax.text(21.5, 62, "v1 (protocolo antigo), um traço por padrão", fontsize=9, color="0.3")
ax.set_yscale("log"); ax.set_xticks(xs, [f"{x}%" for x in xs]); ax.set_xlim(xs[0] - 2, xs[-1] + 14)
ax.set_ylim(0.4, 160)
ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g}%".replace(".", ",")))
ax.set_xlabel("células livres bloqueadas"); ax.set_ylabel("pares que continuam com caminho (escala log)")
ax.set_title("No protocolo antigo os pares sumiam com o bloqueio; no novo, todos ficam")
ax.text(0, -0.2, f"{milhar(SOB.pares_v2.iloc[0])} pares por padrão e nível, {N_MAPAS} mapas, mesmas sementes nos "
        "dois protocolos", transform=ax.transAxes, fontsize=8, color="0.35")
salvar(fig, "orientadora_sobrevivencia_v1_v2")
plt.show()

In [ ]:
pares_v2 = v2("""SELECT mapa, semente, id_problema, path_orig_manh AS caminho FROM v2
                 WHERE n_pivos = 10 AND tipo_degradacao = 'Linear' AND porcentagem_bloqueio = 0.1""")
sobrev_v1 = duckdb.sql(f"""SELECT DISTINCT mapa, semente, id_problema, tipo_degradacao, path_orig_manh AS caminho
    FROM read_csv('{MARCO / "resultados_robustez.csv"}', header = true)
    WHERE n_pivos = 10 AND porcentagem_bloqueio = 0.3""").df()
assert len(sobrev_v1) == int(SOB[SOB.nivel == 0.3].pares_v1.sum())
med_todos = pares_v2.caminho.median()
med_v1 = sobrev_v1.groupby("tipo_degradacao").caminho.median()
fig, ax = plt.subplots(figsize=(7.2, 3.8))
def ecdf(v, **kw):
    v = np.sort(np.clip(v, 1, None)); ax.step(v, np.arange(1, len(v) + 1) / len(v), where="post", **kw)
ecdf(pares_v2.caminho.values, color="#009E73", lw=2.6)
ax.text(med_todos * 1.08, 0.5, f"todos os pares (v2)\nmediana {num(med_todos, 0)}", color="#009E73", fontsize=8.5,
        va="center")
for p in PADRAO_PT:
    ecdf(sobrev_v1[sobrev_v1.tipo_degradacao == p].caminho.values, color=cinzas[p], lw=1.3)
lin = med_v1["Linear"]
ax.annotate(f"sobreviventes do v1 a 30%,\num traço por padrão\n(linear: mediana {num(lin, 0)})",
            xy=(max(lin, 1.2), 0.5), xytext=(1.4, 0.82), fontsize=8.5, color="0.3",
            arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_xscale("log"); ax.set_xlabel("comprimento do caminho ótimo no mapa original, em células (escala log)")
ax.set_ylabel("fração dos pares")
ax.set_title("O v1 só media pares curtos: os longos eram os que sumiam")
plt.show()
display(Markdown(
    f"Mediana do caminho ótimo: **{num(med_todos, 0)} células** em todos os pares; nos sobreviventes do v1 a 30%, "
    + ", ".join(f"{PADRAO_PT[p].lower()} {num(med_v1[p], 0)}" for p in PADRAO_PT) + "."))

In [ ]:
CORES_GRID = {".": "#f6f3ec", "@": "#2b2b2b", "#": "#c9b79c", "o": "#0072B2", "r": "#CC79A7"}
def desenha_grid(ax, g, titulo, marca_passagens):
    chaves = list(CORES_GRID)
    idx = np.vectorize(chaves.index)(g)
    ax.imshow(idx, cmap=ListedColormap([CORES_GRID[c] for c in chaves]), vmin=0, vmax=len(chaves) - 1,
              interpolation="nearest")
    if marca_passagens:
        yy, xx = np.nonzero(g == "o")
        ax.scatter(xx, yy, s=2.2, color=CORES_GRID["o"], marker="o", lw=0)
    ax.set_title(titulo, fontsize=10, loc="left"); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)

def par_de_grids(padrao, nome_fig, titulo):
    g1, g2 = grid(f"den501d_42_{padrao}_v1_30"), grid(f"den501d_42_{padrao}_v2_30")
    r = RESUMO[(RESUMO.mapa == "den501d") & (RESUMO.semente == 42) & (RESUMO.padrao == padrao) &
               (RESUMO.nivel == 0.3)].iloc[0]
    fig, axes = plt.subplots(1, 2, figsize=(9, 5.2))
    desenha_grid(axes[0], g1, f"v1: {int(r.pares_validos_v1)} de {int(r.pares_validos)} pares com caminho", False)
    desenha_grid(axes[1], g2, f"v2: {int(r.pares_validos)} de {int(r.pares_validos)} pares com caminho", True)
    leg = [Patch(color=CORES_GRID["@"], label="obstáculo original"),
           Patch(color=CORES_GRID["#"], label="bloqueada pela degradação"),
           Line2D([], [], color=CORES_GRID["o"], marker="o", ls="", ms=5,
                  label=f"passagem: recusada pela R2 ({int((g2 == 'o').sum())} células)"),
           Patch(color=CORES_GRID["r"], label=f"origem/destino protegido pela R1 ({int((g2 == 'r').sum())})")]
    fig.legend(handles=leg, loc="lower center", ncol=2, fontsize=8.5, bbox_to_anchor=(0.5, -0.06))
    fig.suptitle(titulo, x=0.02, ha="left", fontweight="bold", fontsize=11.5)
    fig.tight_layout()
    salvar(fig, nome_fig)
    plt.show()
    return r

r_lin = par_de_grids("Linear", "exemplo_linear_den501d_v1_v2",
                     "Mesmo mapa, mesma semente, linear a 30%: no v2 as paredes ganham portas")

In [ ]:
display(Markdown(
    f"**O que isso significa.** No v1, os pares que sobravam eram os fáceis: no linear a 30% sobra "
    f"{num(SOB[(SOB.padrao == 'Linear') & (SOB.nivel == 0.3)].taxa_v1.iloc[0] * 100)}% dos pares, com caminho "
    f"mediano de {num(med_v1['Linear'], 0)} células, contra {num(med_todos, 0)} no conjunto todo. No exemplo do "
    f"den501d, o v1 deixa {int(r_lin.pares_validos_v1)} pares; o v2 mantém os {int(r_lin.pares_validos)}, "
    f"abrindo portas nas paredes em vez de descartar quem ficou do outro lado. "
    f"Qualquer conclusão do v1 sobre degradação falava, na prática, de problemas curtos."))

## Ato 4 — O custo do protocolo novo

**Pergunta:** para não partir o mapa, quanto o v2 teve de deformar cada padrão?

Toda vez que um mecanismo tenta bloquear uma célula que dividiria o mapa, a R2 recusa e a célula vira passagem. Meço isso como **células recusadas pela R2 ÷ células bloqueadas**: quanto do obstáculo pretendido virou buraco.

In [ ]:
rej = REJ.assign(nome=REJ.padrao.map(PADRAO_PT))
r30 = rej[rej.nivel == 0.3].sort_values("r2_pct_bloqueadas")
top = r30.iloc[-1]
fig, ax = plt.subplots(figsize=(7.2, 3.4))
for i, (_, linha) in enumerate(r30.iterrows()):
    vals = rej[rej.padrao == linha.padrao].sort_values("nivel")
    ax.plot(vals.r2_pct_bloqueadas, [i] * len(vals), color="0.75", lw=1.2, zorder=1)
    for (_, v), mk in zip(vals.iterrows(), ["o", "s", "D"]):
        ax.scatter(v.r2_pct_bloqueadas, i, marker=mk, s=38, color="#0072B2" if v.nivel == 0.3 else "white",
                   edgecolor="#0072B2", zorder=2)
    ax.text(vals.r2_pct_bloqueadas.max() + 1, i, f"{num(vals.r2_pct_bloqueadas.max(), 0)}%", va="center",
            fontsize=8.5)
ax.set_yticks(range(len(r30)), r30.nome)
ax.set_xlabel("células recusadas pela R2 ÷ células bloqueadas (%)")
ax.legend(handles=[Line2D([], [], marker=m, ls="", color="#0072B2", mfc=f, label=l)
                   for m, f, l in [("o", "white", "10%"), ("s", "white", "20%"), ("D", "#0072B2", "30%")]],
          title="nível", loc="lower right", fontsize=8)
ax.set_title(f"O {top.nome.lower()} é o padrão mais deformado: a 30%, {num(top.r2_pct_bloqueadas, 0)}% do que "
             "ele bloquearia vira passagem")
salvar(fig, "deformacao_por_padrao")
plt.show()

In [ ]:
r_rad = par_de_grids("Radial", "exemplo_radial_den501d_v1_v2",
                     "O radial a 30% no v2 vira queijo suíço: os círculos ganham furos para o mapa não partir")

In [ ]:
baixo = r30.iloc[0]
display(Markdown(
    f"**O que isso significa.** O preço de manter todos os pares é deformar os obstáculos, e o preço varia muito: "
    f"a 30%, de {num(baixo.r2_pct_bloqueadas, 0)}% no {baixo.nome.lower()} a {num(top.r2_pct_bloqueadas, 0)}% no "
    f"{top.nome.lower()}. O bloqueio total continua no alvo (a 30%, entre "
    f"{num(r30.bloqueadas_pct_S.min())}% e {num(r30.bloqueadas_pct_S.max())}% das células livres), mas um radial do "
    f"v2 não é o mesmo obstáculo de um radial do v1: é um radial que nunca fecha um corredor."))

## Ato 5 — O resultado central

**Pergunta:** quando o mapa degrada, quem conserva a vantagem sobre a Manhattan?

Este é o gráfico para levar à orientação. Ele compara o ganho sobre a Manhattan no mapa estático e no degradado (média dos três níveis e dos cinco padrões), para a fórmula e para a Memória-100 nos dois regimes.

In [ ]:
def ganho_deg(regime, e, nivel="todos", tab=G):
    q = tab[(tab.metrica == "absoluta") & (tab.regime == regime) & (tab.nivel == nivel) &
            (tab.estrategia == e)].iloc[0]
    return 1 / q.media_geometrica, 1 / q.ic95_bca_sup, 1 / q.ic95_bca_inf

linhas_ = [("Fórmula", "com_atualizacao", "Fórmula"),
           ("Memória-100", "com_atualizacao", "Memória-100 com atualização"),
           ("Memória-100", "sem_atualizacao", "Memória-100 sem atualização")]
vals = {}
for e, r, nome in linhas_:
    vals[nome] = dict(est=(gm_est[e], *ic_est[e]), deg=ganho_deg(r, e), e=e, r=r)
ret = {k: v["deg"][0] / v["est"][0] for k, v in vals.items()}
assert ret["Fórmula"] > ret["Memória-100 sem atualização"]
assert vals["Fórmula"]["deg"][0] > vals["Memória-100 sem atualização"]["deg"][0]

fig, ax = plt.subplots(figsize=(7.4, 5.2))
for nome, v in vals.items():
    st = estilo(v["e"], v["r"])
    ax.fill_between([0, 1], [v["est"][1], v["deg"][1]], [v["est"][2], v["deg"][2]], color=st["color"],
                    alpha=0.13, lw=0)
    ax.plot([0, 1], [v["est"][0], v["deg"][0]], lw=2.6, ms=8, markevery=[1], **st)
    ax.plot(0, v["est"][0], marker=st["marker"], ms=8, color=st["color"], zorder=3)
estaticos = {}
for nome, v in vals.items():
    estaticos.setdefault((v["e"], round(v["est"][0], 6)), v)
ys_e = 10 ** espalha(np.log10([v["est"][0] for v in estaticos.values()]), 0.06)
for (e, _), ye in zip(estaticos, ys_e):
    ax.text(-0.04, ye, f"{e} {num(estaticos[(e, _)]['est'][0])}×", ha="right", va="center", fontsize=10,
            color=COR[e], fontweight="bold")
ys_d = 10 ** espalha(np.log10([v["deg"][0] for v in vals.values()]), 0.06)
for (nome, v), yd in zip(vals.items(), ys_d):
    ax.text(1.04, yd, f"{num(v['deg'][0])}×  {nome}", ha="left", va="center", fontsize=10, color=COR[v["e"]],
            fontweight="bold")
ax.axhline(1, color=COR["Manhattan"], lw=1, ls=":")
ax.text(0.5, 1.05, "Manhattan = 1×", ha="center", color=COR["Manhattan"], fontsize=9)
ax.set_yscale("log"); ax.set_xlim(-0.75, 1.95)
ax.set_xticks([0, 1], ["mapa estático", "mapa degradado"]); ax.tick_params(axis="x", length=0, labelsize=10.5)
ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g}×".replace(".", ",")))
ax.set_ylabel("ganho sobre a Manhattan (escala log)")
ax.spines["bottom"].set_visible(False)
ax.set_title("A fórmula conserva a vantagem; a memória sem atualização perde quase toda")
ax.text(0, -0.13, "Ganho = expansões da Manhattan ÷ expansões da heurística, no mesmo par e no mesmo estado do mapa.\n"
        f"Média geométrica nos {N_MAPAS} mapas (degradado: {RESUMO.padrao.nunique()} padrões × {RESUMO.nivel.nunique()} "
        "níveis com peso igual); faixas: IC 95% BCa.\nCom atualização = distâncias dos pivôs recalculadas no mapa degradado; "
        "sem = distâncias do mapa original.", transform=ax.transAxes, fontsize=8, color="0.35", va="top")
salvar(fig, "orientadora_ganho_estatico_degradado")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6), sharey=True)
xs = ["estático", "10%", "20%", "30%"]
for ax, regime in zip(axes, REGIME_PT):
    finais = []
    for e in ESTRATEGIAS[1:]:
        y = [gm_est[e]] + [ganho_deg(regime, e, n)[0] for n in ("10", "20", "30")]
        ax.plot(range(4), y, lw=2, ms=6, **estilo(e, regime))
        finais.append((e, y[-1]))
    pos = 10 ** espalha(np.log10([v for _, v in finais]), 0.055)
    for (e, v), yy in zip(finais, pos):
        ax.text(3.12, yy, f"{e.replace('Memória', 'Mem')} {num(v)}×", va="center", fontsize=8.5, color=COR[e])
    ax.axhline(1, color=COR["Manhattan"], lw=1, ls=":")
    ax.set_xticks(range(4), xs); ax.set_xlim(-0.2, 4.1); ax.set_yscale("log")
    ax.set_title(f"memória {REGIME_PT[regime]}", fontsize=10.5)
axes[0].set_ylabel("ganho sobre a Manhattan (escala log)")
axes[0].yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g}×".replace(".", ",")))
for n in ("10", "20", "30"):
    assert all(ganho_deg("sem_atualizacao", m, n)[0] < ganho_deg("com_atualizacao", "Fórmula", n)[0] for m in MEMORIAS)
    assert all(ganho_deg("com_atualizacao", m, n)[0] > 0.9 * gm_est[m] for m in MEMORIAS)
fig.suptitle("Com atualização a memória mantém o ganho; sem atualização ela despenca já a 10%, abaixo da fórmula",
             x=0.02, ha="left", fontweight="bold", fontsize=11.5)
fig.tight_layout()
salvar(fig, "ganho_por_nivel")
plt.show()

In [ ]:
f, m_com, m_sem = vals["Fórmula"], vals["Memória-100 com atualização"], vals["Memória-100 sem atualização"]
display(Markdown(
    f"**O que isso significa.** A fórmula sai de {num(f['est'][0])}× para {num(f['deg'][0])}× "
    f"(conserva {num(100 * ret['Fórmula'], 0)}% do ganho). A Memória-100 com atualização vai de "
    f"{num(m_com['est'][0])}× para {num(m_com['deg'][0])}×: recalcular as distâncias mantém a memória muito à frente. "
    f"Sem atualização ela cai para {num(m_sem['deg'][0])}× (conserva {num(100 * ret['Memória-100 sem atualização'], 0)}%), "
    f"abaixo da fórmula. A fórmula não guarda nada do mapa, então não tem nada a ficar desatualizado."))

## Ato 6 — Por que o δ engana

**Pergunta:** se a fórmula vence sem atualização, por que a Manhattan aparece como "a menos degradada"?

O δ é a perda relativa de cada heurística: expansões no mapa degradado ÷ expansões no mapa original, **dela mesma**. Pense em dois motoristas numa cidade que fecha ruas. Um não conhece a cidade e já dava voltas enormes: com as ruas fechadas, ele dá voltas um pouco maiores. O outro conhecia todos os atalhos: quando os atalhos fecham, a viagem dele piora muito em proporção, mas ele **continua chegando antes**. O δ do primeiro é menor, e mesmo assim ele é o pior motorista. A Manhattan é o primeiro motorista: não tem informação do mapa a perder, então o δ dela mede só o quanto o problema ficou mais difícil.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, metrica, titulo in ((axes[0], "delta", "δ: perda relativa (menor = perdeu menos)"),
                            (axes[1], "absoluta", "ganho sobre a Manhattan no mapa degradado")):
    for i, e in enumerate(ESTRATEGIAS):
        for regime, dx in (("com_atualizacao", -0.12), ("sem_atualizacao", 0.12)):
            if e not in MEMORIAS and regime == "sem_atualizacao":
                continue
            q = G[(G.metrica == metrica) & (G.regime == regime) & (G.nivel == "todos") & (G.estrategia == e)].iloc[0]
            v, lo, hi = q.media_geometrica, q.ic95_bca_inf, q.ic95_bca_sup
            if metrica == "absoluta":
                v, lo, hi = 1 / v, 1 / hi, 1 / lo
            x = i + (dx if e in MEMORIAS else 0)
            st = estilo(e, regime)
            if np.isfinite(lo):
                ax.plot([x, x], [lo, hi], color=st["color"], lw=1.2, ls=st["ls"])
            ax.plot(x, v, marker=st["marker"], ms=8, color=st["color"], mfc=st["markerfacecolor"], mec=st["color"])
    ax.set_xticks(range(len(ESTRATEGIAS)), [e.replace("Memória", "Mem") for e in ESTRATEGIAS], fontsize=8.5)
    ax.set_yscale("log"); ax.set_title(titulo, fontsize=10.5)
    suf = "×" if metrica == "absoluta" else ""
    ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _, suf=suf: f"{v:g}{suf}".replace(".", ",")))
    ax.yaxis.set_minor_formatter(mpl.ticker.FuncFormatter(lambda v, _, suf=suf: f"{v:g}{suf}".replace(".", ",")
                                                          if v in (2, 3, 4, 6) else ""))
axes[0].set_ylabel("δ = expansões degradado ÷ original (escala log)")
axes[1].set_ylabel("ganho (escala log)")
axes[1].legend(handles=[Line2D([], [], marker="o", color=COR["Memória-100"], label="memória com atualização"),
                        Line2D([], [], marker="o", color=COR["Memória-100"], mfc="white", ls="--",
                               label="memória sem atualização")], fontsize=8, loc="upper left")
d_manh = G[(G.metrica == "delta") & (G.nivel == "todos") & (G.estrategia == "Manhattan")].media_geometrica.iloc[0]
d_form = G[(G.metrica == "delta") & (G.nivel == "todos") & (G.estrategia == "Fórmula")].media_geometrica.iloc[0]
assert d_manh < d_form and ganho_deg("sem_atualizacao", "Fórmula")[0] > 1
fig.suptitle("A Manhattan perde menos (δ) porque não tem o que perder; em expansões ela continua a pior",
             x=0.02, ha="left", fontweight="bold", fontsize=11.5)
fig.tight_layout()
salvar(fig, "delta_contra_ganho")
plt.show()

In [ ]:
display(Markdown(
    f"**O que isso significa.** No δ, a Manhattan ({num(d_manh, 3)}) parece mais robusta que a fórmula "
    f"({num(d_form, 3)}). Mas, no mesmo mapa degradado, a fórmula ainda faz {num(ganho_deg('sem_atualizacao', 'Fórmula')[0])}× "
    f"menos expansões que a Manhattan. Por isso o artigo usa as duas métricas: o δ diz quanto cada heurística perdeu; o "
    f"ganho diz quem continua guiando melhor."))

## Ato 7 — Consistência

**Pergunta:** o resultado vale mapa a mapa, ou é efeito da média?

In [ ]:
def matriz(pm, regime):
    t = pm[(pm.metrica == "absoluta") & (pm.nivel == "todos") & (pm.regime == regime)]
    t = 1 / t.pivot(index="mapa", columns="estrategia", values="valor")[ESTRATEGIAS[1:]]
    return t.loc[sorted(t.index, key=lambda m: (GRUPOS.index(categoria(m)), m))]

def heat(ax, t, titulo):
    lv = np.log10(t.values)
    lim = np.abs(lv).max()
    ax.imshow(lv, cmap="PuOr", norm=TwoSlopeNorm(0, -lim, lim), aspect="auto")
    melhor = t.values.argmax(axis=1)
    for i in range(t.shape[0]):
        for j in range(t.shape[1]):
            v = t.values[i, j]
            txt = f"{num(v, 1)}" + (" ★" if j == melhor[i] else "") + (" ✗" if v < 1 else "")
            ax.text(j, i, txt, ha="center", va="center", fontsize=7.5,
                    fontweight="bold" if j == melhor[i] else "normal",
                    color="white" if abs(lv[i, j]) > 0.62 * lim else "black")
    ax.set_xticks(range(t.shape[1]), [c.replace("Memória", "Mem") for c in t.columns], fontsize=8)
    ax.set_yticks(range(t.shape[0]), t.index, fontsize=8)
    ax.tick_params(length=0); ax.set_title(titulo, fontsize=10)
    for s in ax.spines.values(): s.set_visible(False)

t_todos, t_fora = matriz(PM, "sem_atualizacao"), matriz(PMF, "sem_atualizacao")
v_todos = int((t_todos.idxmax(axis=1) == "Fórmula").sum()); v_fora = int((t_fora.idxmax(axis=1) == "Fórmula").sum())
fig, axes = plt.subplots(1, 2, figsize=(10, 6.2))
heat(axes[0], t_todos, f"todos os pares: fórmula melhor em {v_todos} de {len(t_todos)}")
heat(axes[1], t_fora, f"só pares fora do treino do AG: {v_fora} de {len(t_fora)}")
fig.suptitle("Sem atualização, a fórmula é a melhor heurística em todos os mapas (ganho sobre a Manhattan)",
             x=0.02, ha="left", fontweight="bold", fontsize=11.5)
fig.text(0.02, -0.01, "★ melhor do mapa   ✗ pior que a Manhattan (ganho < 1)   roxo: ganha da Manhattan; laranja: perde",
         fontsize=8.5, color="0.35")
fig.tight_layout()
salvar(fig, "consistencia_por_mapa")
plt.show()

In [ ]:
perde = [(m, e) for m in t_todos.index for e in MEMORIAS if t_todos.loc[m, e] < 1]
col_mapas = sorted(COLAPSO.mapa.unique())
ligados = sorted({m for m, _ in perde} & set(col_mapas))
desc = "; ".join(f"{m} ({e}, {num(t_todos.loc[m, e])}×)" for m, e in perde) or "nenhum"
display(Markdown(
    f"Casos em que a memória sem atualização perde para a Manhattan: {desc}. "
    f"Os mapas com colapso de pivô em alguma semente (P8: o primeiro pivô cai num bolsão isolado e a memória vira "
    f"h = 0) são: {', '.join(col_mapas)}. "
    + (f"Entre os casos acima, {', '.join(ligados)} {'está' if len(ligados) == 1 else 'estão'} nessa lista."
       if ligados else "Nenhum dos casos acima está nessa lista.")))
display(COLAPSO)

In [ ]:
display(Markdown(
    f"**O que isso significa.** Não é efeito da média: sem atualização, a fórmula é a melhor heurística em "
    f"{v_todos} de {len(t_todos)} mapas, e o resultado se repete ({v_fora} de {len(t_fora)}) quando eu tiro os pares que o "
    f"algoritmo genético viu durante a síntese. Então a fórmula não está só decorando os pares de treino."))

## Ato 8 — A inversão dos postos

**Pergunta:** a ordem das heurísticas no mapa estático se mantém quando o mapa degrada?

O posto médio de Friedman ordena as seis configurações dentro de cada mapa (1 = melhor) e tira a média entre os mapas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
for ax, metrica, titulo in ((axes[0], "delta", "δ (perda relativa)"), (axes[1], "absoluta", "ganho absoluto")):
    q = POSTOS[(POSTOS.metrica == metrica) & (POSTOS.regime == "sem_atualizacao") & (POSTOS.nivel == "todos")]
    q = q.set_index("estrategia").loc[ESTRATEGIAS]
    for e, lin in q.iterrows():
        st = estilo(e, "sem_atualizacao")
        ax.plot([0, 1], [lin.posto_estatico, lin.posto_degradado], lw=2.2, ms=7, **st)
    pe = espalha(q.posto_estatico.values, 0.28); pd_ = espalha(q.posto_degradado.values, 0.28)
    for (e, lin), a, b in zip(q.iterrows(), pe, pd_):
        ax.text(-0.05, a, f"{e.replace('Memória', 'Mem')} {num(lin.posto_estatico)}", ha="right", va="center",
                fontsize=8.5, color=COR[e])
        ax.text(1.05, b, f"{num(lin.posto_degradado)} {e.replace('Memória', 'Mem')}", ha="left", va="center",
                fontsize=8.5, color=COR[e])
    ax.set_ylim(6.5, 0.5); ax.set_xlim(-0.7, 1.7)
    ax.set_xticks([0, 1], ["estático", "degradado,\nsem atualização"]); ax.set_yticks([])
    for s in ("left", "bottom"): ax.spines[s].set_visible(False)
    ax.tick_params(length=0); ax.set_title(titulo, fontsize=10.5)
qd = POSTOS[(POSTOS.metrica == "delta") & (POSTOS.regime == "sem_atualizacao") & (POSTOS.nivel == "todos")]
rho = qd.posto_estatico.rank().corr(qd.posto_degradado.rank())
inverte = bool(np.isclose(rho, -1))
fig.suptitle("Sem atualização, a ordem estática se inverte" + (" por completo no δ" if inverte else ""),
             x=0.02, ha="left", fontweight="bold", fontsize=11.5)
fig.tight_layout()
salvar(fig, "inversao_postos")
plt.show()
qa = POSTOS[(POSTOS.metrica == "absoluta") & (POSTOS.regime == "sem_atualizacao") & (POSTOS.nivel == "todos")]
qa = qa.set_index("estrategia")
display(Markdown(
    f"No δ, a correlação de postos entre a ordem estática e a degradada sem atualização é {num(rho)}"
    + (": a ordem vira exatamente do avesso." if inverte else ".")
    + f" No ganho absoluto, a fórmula sai do posto {num(qa.posto_estatico['Fórmula'])} para "
    f"{num(qa.posto_degradado['Fórmula'])}, e a Memória-100 do {num(qa.posto_estatico['Memória-100'])} para "
    f"{num(qa.posto_degradado['Memória-100'])}."))

Os diagramas de diferença crítica (Nemenyi) que o pipeline já gera: (a) estático, (b) degradado com atualização, (c) degradado sem atualização. Configurações ligadas por uma barra não se separam estatisticamente.

In [ ]:
for nome in ("fig_b1_cd_absoluta", "fig_b1_cd_delta"):
    display(Markdown(f"`figuras_v2/{nome}.png`"))
    display(Image(filename=str(ANALISE / "figuras_v2" / f"{nome}.png"), width=520))

## Ato 9 — O padrão pesa pouco

**Pergunta:** a forma do bloqueio (linear, radial…) muda quem ganha de quem?

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(11, 3.4), sharey=True)
for ax, p in zip(axes, PADRAO_PT):
    for e, r in (("Fórmula", "com_atualizacao"), ("Memória-100", "com_atualizacao"), ("Memória-100", "sem_atualizacao")):
        q = PN[(PN.metrica == "absoluta") & (PN.regime == r) & (PN.padrao == p) & (PN.estrategia == e)]
        q = q.sort_values("nivel")
        ax.plot(q.nivel.astype(int), 1 / q.valor, lw=1.8, ms=5, **estilo(e, r))
    ax.axhline(1, color=COR["Manhattan"], lw=0.8, ls=":")
    ax.set_title(PADRAO_PT[p], fontsize=10); ax.set_yscale("log")
    nv = sorted(PN.nivel.astype(int).unique())
    ax.set_xticks(nv, [f"{n}%" for n in nv], fontsize=8)
axes[0].set_ylabel("ganho sobre a Manhattan (log)")
pa = PN[PN.metrica == "absoluta"]
for (p, n), q in pa.groupby(["padrao", "nivel"]):
    v = q.set_index(["regime", "estrategia"]).valor
    assert v[("com_atualizacao", "Memória-100")] < v[("com_atualizacao", "Fórmula")] < v[("sem_atualizacao", "Memória-100")]
axes[0].yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g}×".replace(".", ",")))
fig.legend(handles=[Line2D([], [], lw=1.8, ms=5, label=rotulo(e, r), **estilo(e, r))
                    for e, r in (("Fórmula", "com_atualizacao"), ("Memória-100", "com_atualizacao"),
                                 ("Memória-100", "sem_atualizacao"))],
           loc="lower center", ncol=3, fontsize=8.5, bbox_to_anchor=(0.5, -0.08))
fig.suptitle("A ordem é a mesma nos cinco padrões; o que muda é o nível de bloqueio", x=0.02, ha="left",
             fontweight="bold", fontsize=11.5)
fig.tight_layout()
salvar(fig, "padroes_facetas")
plt.show()

In [ ]:
termos = {"C(padrao)": "padrão", "C(nivel)": "nível", "C(estrategia):C(padrao)": "estratégia × padrão",
          "C(estrategia):C(nivel)": "estratégia × nível"}
a2v = ANOVA_V2[("delta", "com_atualizacao")]
eta = pd.DataFrame({"v1 (WPerformance)": ANOVA_V1.loc[list(termos), "eta2_parcial"],
                    "v2": a2v.loc[list(termos), "eta2_parcial"]}).rename(index=termos)
fig, ax = plt.subplots(figsize=(7.2, 2.9))
for i, (t, lin) in enumerate(eta.iterrows()):
    ax.plot([lin.iloc[0], lin.iloc[1]], [i, i], color="0.75", lw=1.5)
    ax.scatter(lin.iloc[0], i, color="white", edgecolor="0.3", s=50, zorder=3, marker="s")
    ax.scatter(lin.iloc[1], i, color="#009E73", s=55, zorder=3)
    ax.text(max(lin) + 0.012, i, f"{num(lin.iloc[0], 3)} → {num(lin.iloc[1], 3)}", va="center", fontsize=8.5)
ax.set_yticks(range(len(eta)), eta.index); ax.invert_yaxis()
ax.set_xlabel("η²ₚ (log δ, memória com atualização, como no v1)")
ax.legend(handles=[Line2D([], [], marker="s", ls="", mfc="white", mec="0.3", label="v1"),
                   Line2D([], [], marker="o", ls="", color="#009E73", label="v2")], loc="lower right", fontsize=8)
ax.set_xlim(0, eta.values.max() + 0.12)
ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g}".replace(".", ",")))
ax.set_title("Com a conectividade preservada, o padrão quase não muda a ordem das estratégias")
salvar(fig, "eta2_v1_v2")
plt.show()
it = INTER.set_index(["metrica", "regime"])
display(it[["F", "eta2_parcial", "p_permutacao", "n_permutacoes", "eta2_parcial_estrategia_nivel"]].round(4))

In [ ]:
e1, e2 = eta["v1 (WPerformance)"], eta["v2"]
pesa = "mais" if e2["nível"] > e2["padrão"] else "menos"
display(Markdown(
    f"**O que isso significa.** A interação estratégia × padrão cai de η²ₚ = {num(e1['estratégia × padrão'], 3)} no v1 "
    f"para {num(e2['estratégia × padrão'], 3)} no v2: boa parte do \"efeito do padrão\" no v1 vinha de cada padrão "
    f"descartar pares diferentes. No v2, o nível de bloqueio pesa {pesa} que o padrão sobre o δ "
    f"(η²ₚ {num(e2['nível'], 3)} contra {num(e2['padrão'], 3)}; no v1 era {num(e1['nível'], 3)} contra "
    f"{num(e1['padrão'], 3)}). A interação continua estatisticamente detectável (p de permutação "
    f"{num(it.loc[('delta', 'com_atualizacao'), 'p_permutacao'], 3)}, o piso com "
    f"{int(it.loc[('delta', 'com_atualizacao'), 'n_permutacoes'])} permutações), mas é pequena: nas facetas, a ordem "
    f"entre fórmula e memória é a mesma nos cinco padrões."))

## Ato 10 — Em construção

Estas análises ainda não existem no pipeline. Deixo as seções marcadas para não esquecer, sem nenhuma estimativa.

### ρ no v2 — em construção
A subotimalidade da fórmula sob degradação: caminho da fórmula ÷ caminho ótimo no mapa degradado (o caminho da Manhattan, que é admissível, dá o ótimo). Os dados já existem (`path_deg_form`, `path_deg_manh`); falta a análise no pipeline, com a mesma escada geométrica.

### Ponto de equilíbrio da atualização — em construção
Com que frequência o mapa pode mudar para que recalcular os pivôs ainda compense: custo do recálculo × expansões economizadas por busca. Precisa de uma medição de tempo com protocolo de benchmark (os tempos atuais não servem, P3).

### Mecanismo do desvio — em construção
Por que a degradação v2 custa caro às heurísticas: quanto do aumento de expansões vem dos desvios forçados pelas passagens da R2. Precisa ligar cada par às passagens que o caminho ótimo degradado atravessa.

### Modelo misto — em construção
O mesmo teste da interação com efeito aleatório de mapa (e de par dentro do mapa), no lugar da ANOVA sobre médias de célula.

## Ato 11 — Fechamento

**Pergunta:** o que mudou do WPerformance (v1) para cá?

In [ ]:
gd = G[(G.nivel == "todos")]
def melhor_glob(metrica, regime, mostra=1):
    q = gd[(gd.metrica == metrica) & (gd.regime == regime)].set_index("estrategia").media_geometrica.sort_values()
    if mostra == 1:
        return q.index[0]
    return q.index[0] + " (" + "; ".join(f"{e} {num(v, 3)}" for e, v in q.iloc[:mostra].items()) + ")"
ETA_GRANDE = 0.14
q1 = np.exp(DELTA_V1.apply(np.log).mean()).sort_values()
v1_menos_degradada = q1.index[0] + " (" + "; ".join(f"{e} {num(v, 3)}" for e, v in q1.iloc[:3].items()) + ")"
v1_inv = POSTOS_V1.posto_estatico.rank().corr(POSTOS_V1.posto_degradado.rank())
sob30 = SOB[SOB.nivel == 0.3]
tab = pd.DataFrame([
    ("Os pares medidos são os mesmos em todos os padrões e níveis",
     "não" if sob30.taxa_v1.nunique() > 1 else "sim", "sim" if (SOB.taxa_v2 == 1).all() else "não"),
    ("A heurística menos degradada (δ, memória com atualização)", v1_menos_degradada,
     melhor_glob("delta", "com_atualizacao", mostra=3)),
    ("A melhor heurística no mapa degradado, em expansões, com atualização", "não medido",
     melhor_glob("absoluta", "com_atualizacao")),
    ("A melhor heurística no mapa degradado, em expansões, sem atualização", "não medido",
     melhor_glob("absoluta", "sem_atualizacao")),
    ("A ordem estática se inverte sob degradação (δ)",
     "por completo" if np.isclose(v1_inv, -1) else ("em parte" if v1_inv < 0 else "não"),
     "por completo, sem atualização" if inverte else "em parte"),
    ("O padrão de degradação muda muito a ordem das estratégias",
     "sim" if e1["estratégia × padrão"] > ETA_GRANDE else "não", "sim" if e2["estratégia × padrão"] > ETA_GRANDE else "não"),
], columns=["conclusão", "v1 (WPerformance)", "v2"])
display(tab.style.hide(axis="index"))

In [ ]:
import re
logs = sorted((V2 / "logs").glob("*.log"))
p11 = {}
for f in logs:
    m = re.search(r"(\d+) divergencias declaradas", f.read_text())
    if m:
        p11[f.stem.split("_")[0]] = int(m.group(1))
nperm = int(INTER.n_permutacoes.iloc[0])
display(Markdown(
    f"Na linha do padrão, \"muda muito\" quer dizer interação com η²ₚ acima de {num(ETA_GRANDE)}, o limiar usual de "
    f"efeito grande; os valores estão no ato 9.\n\n"
    f"**Limitações que continuam valendo**\n\n"
    f"- **Tempo (P3):** os tempos medidos não sustentam nenhuma conclusão; tudo aqui é em expansões.\n"
    f"- **Uma fórmula por mapa (P5):** a síntese de março não é reprodutível; cada mapa tem uma única fórmula.\n"
    f"- **Colapso de pivô (P8):** mantido sem correção, para comparar com o código de março; entra nas médias "
    f"({len(COLAPSO)} combinações de mapa e semente, ato 7).\n"
    f"- **Linha de base (P11):** "
    + (", ".join(f"no `{m}`, {n} problemas da fórmula divergem de março por arredondamento" for m, n in p11.items())
       if p11 else "nenhuma divergência declarada nos logs")
    + " (detalhes em `docs/06-protocolo-v2.md`).\n"
    f"- **Deformação dos padrões:** um padrão do v2 não é o mesmo obstáculo do v1 (ato 4); comparar padrões entre "
    f"protocolos é só qualitativo.\n"
    f"- **Permutação:** com {nperm} permutações, o menor p possível é 1/{nperm + 1}; os p de permutação mostram só "
    f"que nenhuma permutação chegou perto do observado."))

In [ ]:
display(Markdown(f"_Notebook executado em {time.time() - INICIO:.0f} s._"))